# Estrazione features con cellprofiler

In [1]:

from typing import Literal

import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import scanpy as sc
import spatialdata_io
import spatialdata_plot
import squidpy as sq
import spatialdata as sd

from anndata import AnnData
from spatialdata import SpatialData
from spatialdata._core.query.relational_query import _get_unique_label_values_as_index
from spatialdata.datasets import blobs
from spatialdata.models import TableModel
from spatialdata.transformations import Affine, set_transformation
from spatialdata_io.experimental import from_legacy_anndata, to_legacy_anndata
sdata = sd.read_zarr("dati/c26t17/c26t17_def.zarr")

sdata = SpatialData(images = {"C26T17_GFP": sdata["C26T17_GFP"], "C26T17_WGA": sdata["C26T17_WGA"], 
                               "C26T17_dapi": sdata["C26T17_dapi"], "C26T17_merge": sdata["C26T17_merge"]},
                    shapes = {"C26T17": sdata["C26T17"]},
                    table = sdata.table
                    )

sdata
rename_dict = {"aligned": "global"}
sdata.rename_coordinate_systems(rename_dict)

adata = to_legacy_anndata(sdata, include_images=True)


# img = sq.im.ImageContainer(adata.uns["spatial"]["C26T16_merge"]["images"]["hires"], layer="C26T16_merge")
# img.add_img(adata.uns["spatial"]["C26T16_full_image"]["images"]["hires"], layer="C26T16_full_image")
# img.add_img(adata.uns["spatial"]["C26T16_GFP"]["images"]["hires"], layer="C26T16_GFP")
# img.add_img(adata.uns["spatial"]["C26T16_WGA"]["images"]["hires"], layer="C26T16_WGA")
# img.add_img(adata.uns["spatial"]["C26T16_dapi"]["images"]["hires"], layer="C26T16_dapi")

img = sq.im.ImageContainer(sdata["C26T17_merge"]["scale0"].to_dataset()["image"].transpose("y","x", "c").rename({"c":"channel"}).values,
                           layer="C26T17_merge")
img.add_img(sdata["C26T17_GFP"]["scale0"].to_dataset()["image"].transpose("y","x", "c").rename({"c":"channel"}).values, 
            layer="C26T17_GFP")
img.add_img(sdata["C26T17_WGA"]["scale0"].to_dataset()["image"].transpose("y","x", "c").rename({"c":"channel"}).values, 
            layer="C26T17_WGA")
img.add_img(sdata["C26T17_dapi"]["scale0"].to_dataset()["image"].transpose("y","x", "c").rename({"c":"channel"}).values, 
            layer="C26T17_dapi")
img


/tmp/ipykernel_60916/3122921370.py:23: DeprecationWarning: Table accessor will be deprecated with SpatialData version 0.1, use sdata.tables instead.
  table = sdata.table
/tmp/ipykernel_60916/3122921370.py:20: DeprecationWarning: `table` is being deprecated as an argument to `SpatialData.__init__.__init__` in SpatialData version 0.1, switch to `tables` instead.
  sdata = SpatialData(images = {"C26T17_GFP": sdata["C26T17_GFP"], "C26T17_WGA": sdata["C26T17_WGA"],
/home/stefano/.local/lib/python3.10/site-packages/spatialdata/_core/query/relational_query.py:325: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groups_df = table.obs.groupby(by=region_column_name)
/home/stefano/.local/lib/python3.10/site-packages/xarray/core/utils.py:494: FutureWarning: The return type of `Dataset.dims` will be changed to 

ImageContainer[shape=(8212, 10265), layers=['C26T17_GFP', 'C26T17_WGA', 'C26T17_dapi', 'C26T17_merge']]

In [2]:
adata.uns["spatial"]["C26T17_dapi"]["scalefactors"]

{'tissue_hires_scalef': 1.0,
 'tissue_lowres_scalef': 0.3,
 'spot_diameter_fullres': 27.027807508017702}

In [3]:
sdata

SpatialData object with:
├── Images
│     ├── 'C26T17_GFP': MultiscaleSpatialImage[cyx] (3, 8212, 10265), (3, 4102, 5127), (3, 2046, 2558), (3, 1018, 1276), (3, 506, 634)
│     ├── 'C26T17_WGA': MultiscaleSpatialImage[cyx] (3, 8212, 10265), (3, 4102, 5127), (3, 2046, 2558), (3, 1018, 1276), (3, 506, 634)
│     ├── 'C26T17_dapi': MultiscaleSpatialImage[cyx] (3, 8212, 10265), (3, 4102, 5127), (3, 2046, 2558), (3, 1018, 1276), (3, 506, 634)
│     └── 'C26T17_merge': MultiscaleSpatialImage[cyx] (3, 8212, 10265), (3, 4102, 5127), (3, 2046, 2558), (3, 1018, 1276), (3, 506, 634)
├── Shapes
│     └── 'C26T17': GeoDataFrame shape: (1393, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (1393, 19465)
with coordinate systems:
▸ 'global', with elements:
        C26T17_GFP (Images), C26T17_WGA (Images), C26T17_dapi (Images), C26T17_merge (Images), C26T17 (Shapes)

In [4]:
adata.obs

,in_tissue,array_row,array_col,spot_id,region
AAACACCAATAACTGC-1,1,59,19,0,C26T17
AAACAGGGTCTATATT-1,1,47,13,1,C26T17
AAACAGTGTTCCTGGG-1,1,73,43,2,C26T17
AAACCGTTCGTCCAGG-1,1,52,42,3,C26T17
AAACCTCATGAAGTTG-1,1,37,19,4,C26T17
...,...,...,...,...,...
TTGTTGGCAATGACTG-1,1,76,30,1388,C26T17
TTGTTTCACATCCAGG-1,1,58,42,1389,C26T17
TTGTTTCATTAGTCTA-1,1,60,30,1390,C26T17
TTGTTTCCATACAACT-1,1,45,27,1391,C26T17


In [19]:
import pandas as pd
import anndata as ad


for scale in [1.0, 2.0]:
    feature_name = f"features_summary_scale{scale}"
    sq.im.calculate_image_features(
        adata,
        img,
        features="summary",
        key_added=feature_name,
        n_jobs=4,
        scale=scale,
        layer = "C26T17_GFP",
        library_id="C26T17_GFP",
        mask_circle=True
    )
        


# combine features in one dataframe
adata.obsm["features"] = pd.concat(
    [adata.obsm[f] for f in adata.obsm.keys() if "features_summary" in f],
    axis="columns"
)
# make sure that we have no duplicated feature names in the combined table
adata.obsm["features"].columns = ad.utils.make_index_unique(
    adata.obsm["features"].columns
)


  0%|          | 0/1393 [00:00<?, ?/s]/home/stefano/.local/lib/python3.10/site-packages/xarray/core/utils.py:494: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  warnings.warn(
  0%|          | 1/1393 [00:01<42:13,  1.82s/]/home/stefano/.local/lib/python3.10/site-packages/xarray/core/utils.py:494: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  warnings.warn(
/home/stefano/.local/lib/python3.10/site-packages/xarray/core/utils.py:494: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims

In [5]:
adata.obsm["features"]

KeyError: 'features'

In [6]:
img

/home/stefano/.local/lib/python3.10/site-packages/xarray/core/utils.py:494: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  warnings.warn(


ImageContainer[shape=(8212, 10265), layers=['C26T17_GFP', 'C26T17_WGA', 'C26T17_dapi', 'C26T17_merge']]

In [10]:
 #anndata.AnnData.uns ['{spatial_key}']['{library_id}']['scalefactors']
adata.uns["spatial"]["C26T17_GFP"]["scalefactors"]

{'tissue_hires_scalef': 1.0,
 'tissue_lowres_scalef': 0.3,
 'spot_diameter_fullres': 27.027807508017702}

In [11]:
from PIL import Image
import numpy as np
from pathlib import Path

imgs_dir =  "img_C26T17/C26T17_GFP2/"
Path(imgs_dir).mkdir(parents=True, exist_ok=True)

for crop, obs in img.generate_spot_crops(
    adata, library_id="C26T17_GFP",  
    obs_names=adata.obs_names, 
    return_obs=True, 
    as_array=False, 
    spatial_key="spatial",
    spot_diameter_key="spot_diameter_fullres"
    ):
    Image.fromarray((np.array(crop["C26T17_GFP"])).squeeze()).save(
        imgs_dir + f"segments_{obs}.tif"
    )

/home/stefano/.local/lib/python3.10/site-packages/xarray/core/utils.py:494: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  warnings.warn(


In [78]:
adata.obsm["spatial"]

array([[860.03800784, 424.80193806],
       [367.01916054, 779.02180318],
       [963.48684964, 588.75139134],
       ...,
       [658.24729239, 756.89124599],
       [904.82218999, 579.84678845],
       [366.23347314, 756.6293459 ]])